<a href="https://colab.research.google.com/github/mohith-hub/Medical-Abbreviation/blob/main/clg_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
from tensorflow import keras
from packaging import version
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_hub as hub

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/co

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import sys
!{sys.executable} -m pip install Flask

In [9]:
!pip install pyngrok nest_asyncio

In [ ]:
!pip install pyngrok nest_asyncio

In [ ]:
!pip install pyngrok nest_asyncio

In [ ]:
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
import pandas as pd
import tensorflow as tf # Required for tfds.as_numpy_iterator() and other TF operations

# Load the IMDB reviews dataset from TensorFlow Datasets
# We load both train and test splits provided by TFDS.
# as_supervised=False returns examples as dictionaries {'text': ..., 'label': ...}
(ds_train_raw, ds_test_raw), info = tfds.load(
    name="imdb_reviews",
    split=["train", "test"],
    with_info=True,
    as_supervised=False
)

# Function to convert tf.data.Dataset to pandas DataFrame
def tf_dataset_to_df(tf_ds):
    reviews = []
    sentiments = []
    # Iterate through the dataset using as_numpy_iterator to get Python objects
    for example in tf_ds.as_numpy_iterator():
        reviews.append(example['text'].decode('utf-8')) # Decode byte string to regular string
        sentiments.append('positive' if example['label'] == 1 else 'negative')
    return pd.DataFrame({'review': reviews, 'sentiment': sentiments})

# Convert the raw TFDS datasets to pandas DataFrames
df_train_from_tfds = tf_dataset_to_df(ds_train_raw)
df_test_from_tfds = tf_dataset_to_df(ds_test_raw)

# Now, we align with the original notebook's variable names for subsequent cells.
# The original code created train, validation, and test sets from a single 'df'.
# With TFDS, we have a pre-defined train (df_train_from_tfds) and test (df_test_from_tfds).
# We will use df_train_from_tfds to create our 'train_set' and 'valid_set',
# and df_test_from_tfds will be our 'test_set'.

# Split the TFDS training set into training and validation sets (90/10 split)
train_set, valid_set = train_test_split(
    df_train_from_tfds,
    test_size=0.1,
    stratify=df_train_from_tfds['sentiment'],
    random_state=42 # for reproducibility
)

# Assign the TFDS test set as our final test_set
test_set = df_test_from_tfds

# The original `df` variable from the notebook is no longer directly populated in the same way.
# If `df` is used elsewhere, it might need to be defined (e.g., as a concatenation of train/test)
# or handled differently. Based on current context, only `train_set`, `valid_set`, `test_set` are used.

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.G5DUQR_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.G5DUQR_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.G5DUQR_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [ ]:
def df_to_tf_dataset(df, batch_size=32, training=False):
    reviews = tf.constant(df['review'].values, dtype=tf.string)
    labels = tf.constant((df['sentiment'].values == 'positive').astype(int), dtype=tf.int64)

    ds = tf.data.Dataset.from_tensor_slices((reviews, labels))

    if training:
        ds = ds.shuffle(5000)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)
    return ds


In [ ]:
train_set = df_to_tf_dataset(train_set, batch_size=32, training=True)
valid_set = df_to_tf_dataset(valid_set, batch_size=32)
test_set  = df_to_tf_dataset(test_set, batch_size=32)

In [ ]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))


In [ ]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)

Epoch 1/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 988s 1s/step - accuracy: 0.5047 - loss: 0.6934 - val_accuracy: 0.5004 - val_loss: 0.6929
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 959s 1s/step - accuracy: 0.4950 - loss: 0.6931 - val_accuracy: 0.5020 - val_loss: 0.6929


In [ ]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size,mask_zero = True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=5)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1096s 2s/step - accuracy: 0.7243 - loss: 0.5308 - val_accuracy: 0.8012 - val_loss: 0.4127
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1172s 2s/step - accuracy: 0.8571 - loss: 0.3411 - val_accuracy: 0.8660 - val_loss: 0.3019
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1088s 2s/step - accuracy: 0.8785 - loss: 0.2911 - val_accuracy: 0.8792 - val_loss: 0.2941
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1113s 2s/step - accuracy: 0.8884 - loss: 0.2693 - val_accuracy: 0.8700 - val_loss: 0.2896
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1105s 2s/step - accuracy: 0.8980 - loss: 0.2517 - val_accuracy: 0.8788 - val_loss: 0.3015


In [ ]:
text_vec_layer_ragged = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size, ragged=True)
text_vec_layer_ragged.adapt(train_set.map(lambda reviews, labels: reviews))

In [ ]:
embed_size = 128
tf.random.set_seed(42)

model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=5)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1068s 2s/step - accuracy: 0.7062 - loss: 0.5490 - val_accuracy: 0.8476 - val_loss: 0.3513
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1156s 2s/step - accuracy: 0.8609 - loss: 0.3281 - val_accuracy: 0.8724 - val_loss: 0.2906
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1138s 2s/step - accuracy: 0.8804 - loss: 0.2881 - val_accuracy: 0.8768 - val_loss: 0.2899
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1077s 2s/step - accuracy: 0.8928 - loss: 0.2663 - val_accuracy: 0.8732 - val_loss: 0.3054
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1065s 2s/step - accuracy: 0.8989 - loss: 0.2498 - val_accuracy: 0.8752 - val_loss: 0.3076


In [ ]:
import os
import tensorflow_hub as hub
from tensorflow import keras


os.environ["TFHUB_CACHE_DIR"] = "my_tfhub_cache"

class USELayer(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )

    def call(self, inputs):
        return self.use_layer(inputs)

use_layer = USELayer()

text_input = keras.layers.Input(shape=[], dtype="string", name="text")
embedding = use_layer(text_input)
dense = keras.layers.Dense(128, activation="relu")(embedding)
output = keras.layers.Dense(1, activation="sigmoid")(dense)
model = keras.Model(inputs=text_input, outputs=output)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])



model.fit(train_set, validation_data=valid_set, epochs=10)

Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 57s 73ms/step - accuracy: 0.8415 - loss: 0.3715 - val_accuracy: 0.8488 - val_loss: 0.3412
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 50s 71ms/step - accuracy: 0.8592 - loss: 0.3253 - val_accuracy: 0.8464 - val_loss: 0.3461
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 50s 71ms/step - accuracy: 0.8615 - loss: 0.3212 - val_accuracy: 0.8716 - val_loss: 0.3113
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 51s 73ms/step - accuracy: 0.8660 - loss: 0.3125 - val_accuracy: 0.8688 - val_loss: 0.3082
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 49s 69ms/step - accuracy: 0.8695 - loss: 0.3052 - val_accuracy: 0.8700 - val_loss: 0.3086
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 83s 71ms/step - accuracy: 0.8748 - loss: 0.2982 - val_accuracy: 0.8724 - val_loss: 0.3085
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 51s 72ms/step - accuracy: 0.8784 - loss: 0.2895 - val_accuracy: 0.8688 - val_loss: 0.3068
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 55s 78ms/step - accuracy: 0.8822 - loss: 0.2818 - 

In [4]:
import os
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow import keras

# 1. Define the architecture
class USELayer(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )
    def call(self, inputs):
        return self.use_layer(inputs)

# Re-initialize the model since it might have been cleared from memory
use_layer = USELayer()
text_input = keras.layers.Input(shape=[], dtype="string", name="text")
embedding = use_layer(text_input)
dense = keras.layers.Dense(128, activation="relu")(embedding)
output = keras.layers.Dense(1, activation="sigmoid")(dense)
model = keras.Model(inputs=text_input, outputs=output)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# 2. Save the model to a confirmed path
model_path = "/content/movie_sentiment_model.keras"
print(f"Saving model to {model_path}...")
model.save(model_path)

if os.path.exists(model_path):
    print(f"✅ SUCCESS: Model saved at {model_path}")
else:
    print("❌ ERROR: Model file creation failed.")

Saving model to /content/movie_sentiment_model.keras...
✅ SUCCESS: Model saved at /content/movie_sentiment_model.keras


Now, let's create a Flask application to serve a simple web page for movie review sentiment prediction. This app will load the saved model, provide an input form, and display the prediction result.

In [5]:
import sys
import os

# 1. Force install dependencies to ensure they are available in this session
!{sys.executable} -m pip install Flask pyngrok nest_asyncio --quiet

import tensorflow as tf
import tensorflow_hub as hub
from flask import Flask, request, render_template_string
from threading import Thread
import nest_asyncio

# Now import pyngrok after installation
try:
    from pyngrok import ngrok, conf
except ImportError:
    import pkg_resources
    pkg_resources.working_set.add_entry('/usr/local/lib/python3.12/dist-packages')
    from pyngrok import ngrok, conf

# 2. Define Custom Layer for Model Loading
class USELayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )
    def call(self, inputs):
        return self.use_layer(inputs)

app = Flask(__name__)
app.config['ENV'] = 'development'
app.config['DEBUG'] = False

# 3. Load the saved model globally using the verified absolute path
model_path = "/content/movie_sentiment_model.keras"
model = None

if not os.path.exists(model_path):
    print(f"❌ Error: {model_path} not found.")
else:
    try:
        # Load the model with the custom USELayer
        model = tf.keras.models.load_model(
            model_path,
            custom_objects={'USELayer': USELayer}
        )
        print("✅ Model loaded successfully from absolute path!")
    except Exception as e:
        print(f"❌ Error loading model: {e}")

✅ Model loaded successfully from absolute path!


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
